# Community Notes — inspect, clean, and plan
**Peyton Tom · Module 6 · branch `peyton-tom`**

**Question:** How does rating activity vary in this downloaded Ratings file, and what helpfulness choices do raters submit?

**Audience:** a Community Notes product manager reviewing rating participation. This is a first descriptive dashboard for Vision B. It cannot yet measure time to Helpful, establish broad agreement, or decide which notes need more ratings.

**One row:** one submitted rating in the downloaded file. Inspect `(noteId, raterParticipantId)` for repeated pairs; do not assume repeats are mistakes.

**Plan:** inspect all rows and columns, preserve source values, prepare UTC dates, audit full-file duplicates, and verify records and reproducibility. No one-week selection is applied.

**Connection to the team exploration:** `phase1_data_exploration.ipynb` establishes four tables and the request → note → ratings → status funnel. Its first 1,000 rows per table are schema checks, not representative or necessarily matching samples. This notebook starts with the Ratings stage because that real source file is available locally.

**Later dashboard stages:** request coverage needs Requests + Notes joined by `tweetId`; response time needs Notes + Status History joined by `noteId`. Keep every requested post/note in the denominator where appropriate, including unmatched/unresolved records. Aggregate ratings per note before joining to a note-level table to avoid multiplying note counts. Verify join relationships and unmatched keys before adding those metrics. Do not independently join the first 1,000 rows of each table and interpret missing matches as missing coverage.

**Updated after instructor feedback:** the primary dashboard plan in Section 5 now pairs response time with unresolved count and waiting time. The Ratings code below remains supporting exploration, not the response-time calculation.

**File roles:** this notebook explains and inspects the work. `clean_peyton.py` contains the same code cells in the same order for a terminal run. Run either workflow from the repository root; both write the same files to `outputs/peyton-inspection/`. The script's `# %%` sections can also be run interactively in VS Code. Keep code changes synchronized between these two files.

**Class prompting pattern:** name the file, one task, what must stay unchanged, and one check. Example: “In peyton-tom.ipynb, review my UTC timestamp conversion. Keep raw and all selected-week records unchanged. Check that daily counts sum to the selected-week row count.” For the dashboard: “Review Section 5 before building the app. Keep completed response time, unresolved count, and waiting time together; check that the cohort reconciles.”


## Expectations saved before preparation
`evidence/peyton/record_expectations.json` records five AI-assisted expectations derived from original records using independent standard-library date arithmetic. Run `python verify_peyton.py expectations` only for initial setup; it refuses to overwrite existing expectations.


## Peyton Tom — Module 6: inspect before cleaning


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import pandas as pd
from IPython.display import display

source = Path.home() / "Downloads" / "ratings-00007.tsv"
output = Path("outputs/peyton-inspection")
output.mkdir(parents=True, exist_ok=True)
CHUNK_SIZE = 100_000
PREPARE_DATA = True  # Reviewed: preserve every source value and row; add date and flag only.
# Only supply a verified snapshot cutoff. Do not guess it from the latest event.
SNAPSHOT_CUTOFF = None  # Example format: "2026-09-19T00:00:00Z"
cutoff = pd.to_datetime(SNAPSHOT_CUTOFF, utc=True) if SNAPSHOT_CUTOFF else None

source_before = source.stat()
with source.open("rb") as stream:
    source_hash = hashlib.file_digest(stream, "sha256").hexdigest()
columns = pd.read_csv(source, sep="\t", nrows=0).columns.tolist()
required = {"noteId", "raterParticipantId", "createdAtMillis", "helpfulnessLevel"}
assert required.issubset(columns), "Review the source schema before continuing."
print("Source:", source, "| Columns:", len(columns))
print("SHA-256:", source_hash)




## 1. Proposed rules for every column


In [ ]:
binary_columns = [c for c in columns if c in {"agree", "disagree", "helpful", "notHelpful"}
                  or c.startswith("helpful") or c.startswith("notHelpful")]
binary_columns = [c for c in binary_columns if c != "helpfulnessLevel"]
id_columns = [c for c in ["noteId", "raterParticipantId", "ratedOnTweetId", "suggestionId"] if c in columns]
category_columns = binary_columns + [c for c in ["helpfulnessLevel", "version", "ratingSourceBucketed"] if c in columns]
known_helpfulness = {"HELPFUL", "SOMEWHAT_HELPFUL", "NOT_HELPFUL"}
rules = []
for col in columns:
    if col in binary_columns:
        rule, check = "Preserve; flag values other than 0 or 1", "Missing count and unexpected binary count"
    elif col in id_columns:
        rule, check = "Keep original text; flag blank/format/sentinel concerns", "Original ID strings unchanged"
    elif col == "createdAtMillis":
        rule, check = "Preserve milliseconds; optionally add UTC date", "Integer syntax, parse failures, nonpositive values, cutoff if supplied"
    elif col == "helpfulnessLevel":
        rule, check = "Preserve exact categories; flag unexpected labels", "Full-file category counts reconcile"
    elif col == "suggestion":
        rule, check = "Preserve text and blanks; do not infer missing means bad rating", "Empty/whitespace counts and nonempty examples"
    else:
        rule, check = "Preserve; review observed values before proposing changes", "Missing, whitespace, and observed examples"
    rules.append({"column": col, "proposed_rule": rule, "verification": check})
rules = pd.DataFrame(rules)
display(rules)




## 2. Inspect the entire file and flag suspicious values


In [ ]:
def flag_chunk(raw):
    flags = pd.DataFrame(index=raw.index)
    for col in columns:
        values = raw[col]
        flags[col + "__blank"] = values.str.strip().eq("")
        if col in binary_columns:
            flags[col + "__unexpected_binary"] = ~values.isin(["0", "1"]) & ~flags[col + "__blank"]
    for col in id_columns:
        if col == "raterParticipantId":
            # Opaque participant IDs: no undocumented length/format requirement.
            flags[col + "__surrounding_space"] = raw[col].ne(raw[col].str.strip())
        else:
            flags[col + "__format_or_sentinel"] = ~raw[col].str.fullmatch(r"[0-9]+", na=False) & raw[col].ne("")
    values = raw["createdAtMillis"]
    integer = values.str.fullmatch(r"-?[0-9]+", na=False)
    numbers = pd.to_numeric(values.where(integer), errors="coerce")
    dates = pd.to_datetime(numbers.where(numbers.gt(0)), unit="ms", utc=True, errors="coerce")
    flags["invalid_rating_date"] = dates.isna()
    flags["nonpositive_timestamp"] = numbers.le(0).fillna(False)
    if cutoff is not None:
        flags["date_after_cutoff"] = dates.gt(cutoff)
    flags["unexpected_helpfulness"] = ~raw["helpfulnessLevel"].isin(known_helpfulness) & raw["helpfulnessLevel"].ne("")
    flags["repeated_pair_within_chunk"] = raw.duplicated(["noteId", "raterParticipantId"], keep=False)
    return flags, dates

profile = {c: {"empty": 0, "whitespace_only": 0, "outer_whitespace": 0, "examples": []} for c in columns}
frequencies = {c: Counter() for c in category_columns}
overflow = set()
flag_counts = Counter()
review_parts = []
review_count = 0
rows = 0
date_min = None
date_max = None
for raw in pd.read_csv(source, sep="\t", dtype="string", keep_default_na=False, chunksize=CHUNK_SIZE):
    flags, dates = flag_chunk(raw)
    flag_counts.update({c: int(flags[c].sum()) for c in flags})
    for col in columns:
        values = raw[col]
        stripped = values.str.strip()
        profile[col]["empty"] += int(values.eq("").sum())
        profile[col]["whitespace_only"] += int((stripped.eq("") & values.ne("")).sum())
        profile[col]["outer_whitespace"] += int(values.ne(stripped).sum())
        examples = profile[col]["examples"]
        if len(examples) < 3:
            for value in values.loc[values.ne("")].head(100).unique():
                if value not in examples and len(examples) < 3:
                    examples.append(value)
        if col in frequencies and col not in overflow:
            counts = values.value_counts().to_dict()
            if len(frequencies[col].keys() | counts.keys()) <= 1000:
                frequencies[col].update(counts)
            else:
                overflow.add(col)
                frequencies[col].clear()
    valid_dates = dates.dropna()
    if not valid_dates.empty:
        low, high = valid_dates.min(), valid_dates.max()
        date_min = low if date_min is None else min(date_min, low)
        date_max = high if date_max is None else max(date_max, high)
    if review_count < 50:
        # A blank optional suggestion alone is not a suspicious rating.
        concerns = flags.drop(columns=[c for c in flags if c.endswith("__blank") and c not in
            ["noteId__blank", "raterParticipantId__blank", "createdAtMillis__blank", "helpfulnessLevel__blank"]])
        chosen = raw.loc[concerns.any(axis=1)].head(50 - review_count).copy()
        chosen.insert(0, "source_row", chosen.index + 1)
        if not chosen.empty:
            review_parts.append(chosen)
            review_count += len(chosen)
    rows += len(raw)
    if rows % 1_000_000 == 0:
        print(f"Inspected {rows:,} rows")

profile_table = pd.DataFrame.from_dict(profile, orient="index").rename_axis("column").reset_index()
profile_table["rows_inspected"] = rows
profile_table["examples"] = profile_table["examples"].map(json.dumps)
decisions = rules.merge(profile_table, on="column", validate="one_to_one")
flags_table = pd.DataFrame(sorted(flag_counts.items()), columns=["flag", "rows_flagged"])
category_table = pd.DataFrame([{"column": c, "value": v, "count": count}
                              for c, counts in frequencies.items() for v, count in counts.items()],
                             columns=["column", "value", "count"])
for col in category_columns:
    if col not in overflow:
        assert sum(frequencies[col].values()) == rows

decisions.to_csv(output / "column_rules_and_evidence.csv", index=False)
flags_table.to_csv(output / "flag_counts.csv", index=False)
category_table.to_csv(output / "category_counts.csv", index=False)
if review_parts:
    pd.concat(review_parts).to_csv(output / "flagged_sample.csv", index=False)
else:
    pd.DataFrame(columns=["source_row"] + columns).to_csv(output / "flagged_sample.csv", index=False)
manifest = {"source": str(source), "sha256": source_hash, "rows": rows, "columns": columns,
            "source_bytes": source_before.st_size, "earliest_rating_utc": str(date_min),
            "latest_rating_utc": str(date_max), "snapshot_cutoff": SNAPSHOT_CUTOFF,
            "incomplete_category_counts": sorted(overflow), "pandas_version": pd.__version__,
            "duplicate_check_scope": "Repeated note/rater pairs within chunks only; no removal"}
(output / "inspection_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(f"Inspection: {rows:,} rows; 0 source values changed; 0 rows removed.")
print("Observed rating dates:", date_min, "to", date_max)
display(decisions)
display(flags_table.loc[flags_table["rows_flagged"].gt(0)])
if cutoff is None:
    print("Unresolved: no verified snapshot cutoff; future-date check was not performed.")




## 3. Review the evidence before preparing the analysis table


In [ ]:
if PREPARE_DATA:
    # Stream the full file to disk; never concatenate 26 million rows in memory.
    temporary = output / "ratings_prepared.csv.partial"
    final = output / "ratings_prepared.csv"
    written = 0
    invalid_dates = 0
    sample_parts = []
    sample_count = 0
    for raw in pd.read_csv(source, sep="\t", dtype="string", keep_default_na=False, chunksize=CHUNK_SIZE):
        integer = raw["createdAtMillis"].str.fullmatch(r"-?[0-9]+", na=False)
        numbers = pd.to_numeric(raw["createdAtMillis"].where(integer), errors="coerce")
        dates = pd.to_datetime(numbers.where(numbers.gt(0)), unit="ms", utc=True, errors="coerce")
        prepared = raw.copy()
        prepared["rating_date_utc"] = dates
        prepared["invalid_rating_date"] = dates.isna()
        assert len(prepared) == len(raw)
        assert prepared[columns].equals(raw), "An original field was changed"
        prepared.to_csv(temporary, mode="w" if written == 0 else "a", header=written == 0, index=False)
        if sample_count < 50:
            sample = prepared.head(50 - sample_count)
            sample_parts.append(sample)
            sample_count += len(sample)
        invalid_dates += int(dates.isna().sum())
        written += len(prepared)
    assert written == rows
    if written == 0:
        pd.DataFrame(columns=columns + ["rating_date_utc", "invalid_rating_date"]).to_csv(temporary, index=False)
    # Original source must still be the inspected file.
    with source.open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == source_hash
    temporary.replace(final)
    if sample_parts:
        pd.concat(sample_parts).to_csv(output / "prepared_sample.csv", index=False)
    changes = pd.DataFrame([
        {"step": "Load all source fields as text", "rows_before": rows, "rows_after": rows,
         "original_values_changed": 0, "details": "No row selection or ID conversion"},
        {"step": "Add rating_date_utc", "rows_before": rows, "rows_after": written,
         "original_values_changed": 0, "details": f"Milliseconds represented as UTC in NEW column; {invalid_dates} missing derived dates"},
        {"step": "Add invalid_rating_date flag", "rows_before": rows, "rows_after": written,
         "original_values_changed": 0, "details": "Flags retained; no exclusions or category changes"},
    ])
    changes.to_csv(output / "transformation_report.csv", index=False)
    display(changes)
else:
    print("Inspection only. No cleaned dataset was written; review the rules and flags first.")
assert source.stat().st_size == source_before.st_size
assert source.stat().st_mtime_ns == source_before.st_mtime_ns


## Full-file verification
The following checks compare all source columns with serialized prepared data, audit duplicate rows and IDs across the entire file, and compare five records with expectations saved before preparation. Raw strings, optional blanks, and sentinel values are retained.


In [ ]:
from verify_peyton import duplicates, verify
duplicates()
verify()


## 5. Dashboard plan — incorporate the Phase 1 feedback
**Main question:** How quickly do notes reach the chosen status, and how many are still waiting at the same observation cutoff?

The instructor identified selection bias: the median among notes that reach a status excludes those that never reach it. Keep the completed-time median, but always show the unresolved count and waiting time beside it. The Ratings analysis above is supporting exploration; it does not yet calculate these outcomes.

### Define the measurement before coding
- **Unit:** one note, identified by `noteId`; do not count rating rows as notes.
- **Cohort:** notes created in the selected date range, with observation through one explicit dataset cutoff. Include notes without ratings or without an observed outcome.
- **Target:** provisionally first Helpful, following `objectives.md`. Confirm the endpoint used in the team's response-time calculation before comparing results. First non-NMR, first Helpful, and current status are different measures.
- **Completed response time:** verified first-target timestamp minus note creation, for notes reaching that target on or before the cutoff. Report hours and the contributing note count.
- **Unresolved count:** notes with sufficient outcome coverage that have not reached the target by the cutoff. Label them “Not yet reached Helpful” if Helpful is the target; this is not necessarily the same as currently Needs More Ratings.
- **Waiting time:** cutoff minus creation for those unresolved notes. Report median age in hours and their count. These are ongoing waits, not completed response times.
- **Unknown outcome:** missing/unmatched history or unusable timestamps must be shown separately; do not silently treat these as unresolved or drop them.

### Layout
| View | Contents |
|---|---|
| Scope and filters | Note creation-date range; optional note-type filter only if verified in Notes; display fixed observation cutoff |
| KPI 1 | Median completed time to target (hours), with completed-note count |
| KPI 2 | Unresolved-note count, with total cohort size and unresolved share |
| KPI 3 beside KPI 2 | Median unresolved waiting time (hours), with usable waiting-time count |
| Data-quality note | Unknown-outcome count and invalid/missing creation times |
| Chart 1 | Completed, unresolved, and unknown counts by creation week; show cohort size |
| Chart 2 | Distribution of unresolved waiting times; explicit hour/day units |

Use the same note cohort and cutoff for every view. Do not filter the cohort to successful notes or to notes that received ratings. If no notes completed, show the completed median as “N/A,” not zero. If none are unresolved, show count zero and waiting median “N/A.” A newer cohort has had less opportunity to reach the target; label this when comparing creation weeks.

### Data needed and checks
Use the full Notes cohort and left-join appropriate Status History by `noteId`, validating one row per note and reporting unmatched IDs. Verify whether available fields actually identify the first occurrence of the chosen target. A current-status timestamp must not be substituted for first Helpful, and first non-NMR may be Not Helpful. If the endpoint cannot be recovered, mark the metric unavailable or explicitly agree on another endpoint.

Ratings can later add rating volume after aggregation to one row per note; they cannot supply the outcome timestamp. Do not join independently selected first-1,000-row samples and interpret unmatched rows as an unresolved backlog.

**Independent calculation check:** at a fixed cutoff, suppose two notes reached the target in 2 and 4 hours and three have not reached it after 24, 48, and 72 hours. Expected display: completed median **3 hours (n=2)**, unresolved **3 of 5 notes (60%)**, unresolved median wait **48 hours (n=3)**. The low completed median does not eliminate the backlog.

**Reconciliation:** cohort count = completed + unresolved + unknown outcome. Report how many rows support each time calculation and flag negative durations. A snapshot reveals current backlog size and age; claiming it is *growing* requires comparable historical cutoffs or sufficient event history to reconstruct them.

**Implementation status:** this is the revised dashboard plan. The notebook currently cleans the available Ratings file only. Notes and Status History, endpoint verification, and real calculations are still needed before building the response-time dashboard.

### What the saved Phase 1 output actually establishes
The status-history schema printed in `phase1_data_exploration.ipynb` includes `noteId`, `createdAtMillis`, `timestampMillisOfFirstNonNMRStatus`, `firstNonNMRStatus`, and `currentStatus`. The first ten displayed records have current status `NEEDS_MORE_RATINGS` and missing first non-NMR status. Keep these records; missing an outcome timestamp does not justify dropping the note.

This table includes creation time itself, so a basic **time to first non-NMR status** and unresolved-wait analysis does not require a Notes join. That endpoint includes both Helpful and Not Helpful; it must be named explicitly and must not be relabeled time to first Helpful. The saved schema does not establish a general first-Helpful timestamp. Notes is needed for additional classifications and for checking the intended cohort's coverage.

The notebook references `data/noteStatusHistory-00000.tsv`, but that file is not in this checkout. Saved HTML/text previews are not the full 1,000-row sample or the original dataset. Full-data calculations require that TSV plus a verified observation cutoff. Do not use the maximum observed event time as if it were a confirmed download cutoff.


## Submission evidence
See `CHECKLIST-peyton.md` and `evidence/peyton/`. The source download date and snapshot cutoff remain unknown; first-Helpful outcome data is still unavailable. All original rows and columns are preserved. No claim is made that an AI-assisted record check was performed manually by the student.
